

---

# 1097. Game Play Analysis V

## Table: `Activity`

| Column Name  | Type |
|--------------|------|
| player_id    | int  |
| device_id    | int  |
| event_date   | date |
| games_played | int  |

- `(player_id, event_date)` is the **primary key** of this table.  
- Each row represents a record of a player who logged in and played a certain number of games (possibly 0) before logging out on a given day using a specific device.

---

## Problem Description

- The **install date** of a player is defined as the first day they logged into the game.  
- The **Day 1 retention** for a given date `X` is defined as:

\[
\text{Day1\_retention}(X) = \frac{\text{Number of players whose install date is X and logged in again on X+1}}{\text{Number of players whose install date is X}}
\]

- The result should be rounded to **2 decimal places**.

---

## Task

Write an SQL query to report, for each install date:
1. The number of players who installed the game on that date (`installs`)  
2. The **Day 1 retention** value  

---

## Example

### Input: `Activity` table

| player_id | device_id | event_date | games_played |
|-----------|-----------|------------|--------------|
| 1         | 2         | 2016-03-01 | 5            |
| 1         | 2         | 2016-03-02 | 6            |
| 2         | 3         | 2017-06-25 | 1            |
| 3         | 1         | 2016-03-01 | 0            |
| 3         | 4         | 2016-07-03 | 5            |

### Output: Result table

| install_dt | installs | Day1_retention |
|------------|----------|----------------|
| 2016-03-01 | 2        | 0.50           |
| 2017-06-25 | 1        | 0.00           |

---

## Explanation

- Players `1` and `3` installed the game on **2016-03-01**.  
  - Only player `1` logged back in on **2016-03-02**.  
  - Day 1 retention = \( \frac{1}{2} = 0.50 \).  

- Player `2` installed the game on **2017-06-25**.  
  - They did not log back in on **2017-06-26**.  
  - Day 1 retention = \( \frac{0}{1} = 0.00 \).  

---

Would you like me to also create a **SQL query template** (with placeholders but no solution logic) so you can practice filling it in yourself?

In [0]:
import datetime
from pyspark.sql.types import StructType, StructField, IntegerType, DateType

activity_schema = StructType([
    StructField("player_id", IntegerType(), False),
    StructField("device_id", IntegerType(), True),
    StructField("event_date", DateType(), False),
    StructField("games_played", IntegerType(), True)
])

activity_data = [
    (1, 2, datetime.date(2016, 3, 1), 5),
    (1, 2, datetime.date(2016, 3, 2), 6),
    (2, 3, datetime.date(2017, 6, 25), 1),
    (3, 1, datetime.date(2016, 3, 1), 0),
    (3, 4, datetime.date(2016, 7, 3), 5)
]

activity_df = spark.createDataFrame(activity_data, schema=activity_schema)
activity_df.show()
activity_df.createOrReplaceTempView("Activity")


In [0]:
%sql
with cte as (
  select player_id , event_date  , 
  rank()over(partition by player_id order by event_date asc) as rnk , 
  lead(event_date,1)over(partition by player_id order by event_date asc) as next_login , 
  
  *
  from Activity
)
Select 
event_date  as install_date,  count(*) as installs 
,
round(sum
(case when datediff(next_login,event_date) = 1 then 1 else 0 end)/
count(* )
,2) as result 

from cte
where rnk = 1
group by event_date